<a href="https://colab.research.google.com/github/AyseNil-star/AyseNil-star/blob/main/week02_numbering_and_repertoires_ipynb_json.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 2 — Numbering, Germlines, and Repertoires

**Computational Antibody Engineering**

Last week I handed you six CDR sequences and told you they were trastuzumab's CDRs. You used them,
found the paratope, and moved on.

I did not tell you which **numbering scheme** those CDRs came from. There are several, they disagree,
and the disagreement is not small. Today you find out what I gave you, what the alternatives would
have given you, and why this is one of the most common sources of silent error in antibody work.

The question for today:

> **If two people say "mutate H52", are they talking about the same residue?**

Not necessarily.

---
### Today's tool
**ANARCI** — **Antibody Numbering and Antigen Receptor ClassIfication**. It takes an antibody sequence,
finds the variable domain, numbers it under whichever scheme you ask for, and tells you which germline
genes it came from. It is the workhorse underneath most of the OPIG tools you will meet later, so it is worth knowing directly rather than through a web form.

## 0. Setup

ANARCI needs **HMMER**, a profile hidden-Markov-model search package, because that is how it
recognises a variable domain. HMMER is a compiled binary, not a Python package, so we install it with
`apt-get` first and then install ANARCI with `pip`.

This cell takes **1–2 minutes** — considerably longer than week 1. Start it now and read the next
section while it runs.

In [ ]:
# HMMER first (compiled binary), then ANARCI (Python) which calls it.
!apt-get install -qq hmmer > /dev/null 2>&1
%pip install -q anarci biopython pandas

import warnings
warnings.filterwarnings("ignore")

import shutil
assert shutil.which("hmmscan"), "hmmscan not found - re-run this cell"

from anarci import anarci, number
import pandas as pd
import urllib.request, os

print("ANARCI ready. hmmscan at:", shutil.which("hmmscan"))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 65.5 MB/s eta 0:00:00
ANARCI ready. hmmscan at: /usr/bin/hmmscan


## 1. The reveal

Here is trastuzumab's variable heavy (VH) and variable light (VL) domain, same molecule as last week.

And here are the CDRs I gave you in week 1. Keep them on screen — we are about to reproduce them,
and then fail to reproduce them.

In [ ]:
VH = ("EVQLVESGGGLVQPGGSLRLSCAASGFNIKDTYIHWVRQAPGKGLEWVARIYPTNGYTRYADSVKG"
      "RFTISADTSKNTAYLQMNSLRAEDTAVYYCSRWGGDGFYAMDYWGQGTLVTVSS")
VL = ("DIQMTQSPSSLSASVGDRVTITCRASQDVNTAVAWYQQKPGKAPKLLIYSASFLYSGVPSRFSGSR"
      "SGTDFTLTISSLQPEDFATYYCQQHYTTPPTFGQGTKVEIK")

WEEK1_CDRS = {
    "L1": "RASQDVNTAVA", "L2": "SASFLYS",  "L3": "QQHYTTPPT",
    "H1": "DTYIH",       "H2": "RIYPTNGYTRYADSVKG", "H3": "WGGDGFYAMDY",
}

print(f"VH: {len(VH)} residues")
print(f"VL: {len(VL)} residues")
print("\nWeek 1 CDRs:")
for k, v in WEEK1_CDRS.items():
    print(f"  {k}: {v}")

VH: 120 residues
VL: 107 residues

Week 1 CDRs:
  L1: RASQDVNTAVA
  L2: SASFLYS
  L3: QQHYTTPPT
  H1: DTYIH
  H2: RIYPTNGYTRYADSVKG
  H3: WGGDGFYAMDY


## 2. Numbering one sequence three ways

A **numbering scheme** assigns a position label to every residue in a variable domain, so that
"position 52" means the structurally equivalent place in *any* antibody. That is the whole purpose:
antibodies differ in length, so raw sequential counting is useless for comparison.

The three you must know:

| Scheme | Basis | Still used because |
|---|---|---|
| **Kabat** (1970s) | sequence variability | enormous legacy literature, many patents |
| **Chothia** | structural loop positions | canonical CDR conformations are defined on it |
| **IMGT** | germline gene alignment | current standard; databases and most modern tools use it |

They are *different conventions over the same molecule.* None is wrong. They disagree.

Let's number the same VH under each and just look at the length of the result.

In [ ]:
for scheme in ["imgt", "kabat", "chothia"]:
    numbering, chain_type = number(VH, scheme=scheme)
    occupied = [(p, a) for (p, ic), a in numbering if a != "-"]
    highest = max(p for (p, ic), a in numbering if a != "-")
    print(f"{scheme:<9} chain={chain_type}  {len(occupied)} residues numbered, "
          f"highest position = {highest}")

imgt      chain=H  120 residues numbered, highest position = 128
kabat     chain=H  120 residues numbered, highest position = 113
chothia   chain=H  120 residues numbered, highest position = 113


**Read that carefully.** The same 119 amino acids. IMGT runs the position counter up to 128,
Kabat and Chothia stop at 113.

The schemes are not just re-labelling — they reserve *different numbers of slots*. IMGT allocates
fixed positions for structural elements whether or not a given antibody fills them, so most antibodies
have gaps. Kabat numbers what is there and uses insertion codes when it runs out of room.

This is why you cannot convert between schemes with arithmetic. There is no offset to add.

## 3. The H52 problem

Now, Let's ask each scheme what amino acid sits at **heavy chain position 52**,
and at position 33.

In [ ]:
def residue_at(seq, scheme, position):
    """What amino acid is at this position under this scheme? None if the slot is empty."""
    numbering, _ = number(seq, scheme=scheme)
    for (p, insertion_code), aa in numbering:
        if p == position and insertion_code == " " and aa != "-":
            return aa
    return None


print(f"{'scheme':<10}{'H33':<8}{'H52':<8}{'H100'}")
print("-" * 34)
for scheme in ["imgt", "kabat", "chothia"]:
    row = [residue_at(VH, scheme, p) for p in (33, 52, 100)]
    print(f"{scheme:<10}" + "".join(f"{str(r):<8}" for r in row))

scheme    H33     H52     H100
----------------------------------
imgt      None    W       A       
kabat     Y       Y       F       
chothia   Y       Y       F       


### Sit with this for a second.

- **H52 is tryptophan under IMGT and tyrosine under Kabat and Chothia.** Same molecule, same position
  label, different amino acid.
- **H33 does not exist under IMGT** — that slot is empty in this antibody — but it is a tyrosine under
  Kabat and Chothia.

So the answer to today's question is **no**. "Mutate H52" is ambiguous unless the scheme is stated.

This is not a pedantic edge case. It is a routine cause of real errors: a mutation ordered against
the wrong residue, a literature position mapped onto the wrong structure, a dataset merged from two
sources that used different conventions. **Whenever you record an antibody position, record the scheme
with it.**

## 4. Where do the CDRs actually start and stop?

Each scheme also carries its own **CDR definition** — a range of positions declared to be the CDR.
These are lookup tables, not calculations. You look them up; you do not derive them.

Note the asymmetry below: IMGT uses one set of ranges for both chains, while Kabat and Chothia use
different ranges for heavy and light.

In [ ]:
# CDR position ranges, by scheme and chain type. These are conventions - look them up, don't derive them.
REGIONS = {
    "imgt": {
        "H": {"CDR1": (27, 38), "CDR2": (56, 65), "CDR3": (105, 117)},
        "L": {"CDR1": (27, 38), "CDR2": (56, 65), "CDR3": (105, 117)},
    },
    "kabat": {
        "H": {"CDR1": (31, 35), "CDR2": (50, 65), "CDR3": (95, 102)},
        "L": {"CDR1": (24, 34), "CDR2": (50, 56), "CDR3": (89, 97)},
    },
    "chothia": {
        "H": {"CDR1": (26, 32), "CDR2": (52, 56), "CDR3": (95, 102)},
        "L": {"CDR1": (24, 34), "CDR2": (50, 56), "CDR3": (89, 97)},
    },
}


def get_cdrs(seq, scheme):
    """Return {CDR name: sequence} for a variable domain under one scheme."""
    numbering, chain_type = number(seq, scheme=scheme)
    ctype = "H" if chain_type == "H" else "L"     # ANARCI returns K or L for light chains
    occupied = [((p, ic), a) for (p, ic), a in numbering if a != "-"]
    out = {}
    for name, (lo, hi) in REGIONS[scheme][ctype].items():
        out[name] = "".join(a for (p, ic), a in occupied if lo <= p <= hi)
    return out


rows = []
for label, seq in [("VH", VH), ("VL", VL)]:
    for scheme in ["imgt", "kabat", "chothia"]:
        c = get_cdrs(seq, scheme)
        rows.append({
            "domain": label, "scheme": scheme,
            "CDR1": c["CDR1"], "len1": len(c["CDR1"]),
            "CDR2": c["CDR2"], "len2": len(c["CDR2"]),
            "CDR3": c["CDR3"], "len3": len(c["CDR3"]),
        })

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 30)
pd.DataFrame(rows)

,domain,scheme,CDR1,len1,CDR2,len2,CDR3,len3
0,VH,imgt,GFNIKDTY,8,IYPTNGYT,8,SRWGGDGFYAMDY,13
1,VH,kabat,DTYIH,5,RIYPTNGYTRYADSVKG,17,WGGDGFYAMDY,11
2,VH,chothia,GFNIKDT,7,YPTNGY,6,WGGDGFYAMDY,11
3,VL,imgt,QDVNTA,6,SAS,3,QQHYTTPPT,9
4,VL,kabat,RASQDVNTAVA,11,SASFLYS,7,QQHYTTPPT,9
5,VL,chothia,RASQDVNTAVA,11,SASFLYS,7,QQHYTTPPT,9


### YOUR TURN — Question 1

Look at the table above and answer in the text cell below.

1. Compare **CDR-H1** across the three schemes. Write out the three sequences. How many residues do
   all three agree on? (Kabat's and Chothia's H1 barely overlap — check for yourself.)
2. Compare **CDR-H2**. Kabat's is how many times longer than Chothia's?
Kabat length: 17 residues (RIYPTNGYTRYADSVKG)   Chothia length: 6 residues (YPTNGY)   Ratio: Kabat is $\frac{17}{6} \approx 2.83$
3. Compare **CDR-L1**. Which two schemes agree exactly, and which is the odd one out?
Agree exactly: Kabat and Chothia both define CDR-L1 as RASQDVNTAVA (length 11).   Odd one out: IMGT, which defines it as QDVNTA (length 6).
4. Now go back to week 1 at the top. **Which scheme did I give you in week 1?** Justify it from
   the table — find the row that matches all six.

In [ ]:
# Run this AFTER you have answered, to check yourself.
for scheme in ["imgt", "kabat", "chothia"]:
    h, l = get_cdrs(VH, scheme), get_cdrs(VL, scheme)
    derived = {"H1": h["CDR1"], "H2": h["CDR2"], "H3": h["CDR3"],
               "L1": l["CDR1"], "L2": l["CDR2"], "L3": l["CDR3"]}
    matches = sum(derived[k] == WEEK1_CDRS[k] for k in WEEK1_CDRS)
    verdict = "  <-- this is what week 1 used" if matches == 6 else ""
    print(f"{scheme:<9} matches {matches}/6 of the week 1 CDRs{verdict}")

imgt      matches 1/6 of the week 1 CDRs
kabat     matches 6/6 of the week 1 CDRs  <-- this is what week 1 used
chothia   matches 4/6 of the week 1 CDRs


So week 1 was Kabat, and I never said so. That is exactly how this error happens in practice —
not through anyone lying, but through a convention being so familiar to whoever wrote it down that
they forgot it was a choice.

Notice also what changes if you had used IMGT instead. IMGT's CDR-H3 is `SRWGGDGFYAMDY` — it includes
the `SR` that Kabat leaves in the framework. If you had told an inverse-folding model to
"redesign CDR-H3", those two extra residues would or would not be in play depending on a decision you
may not have realised you were making.

## 5. Insertion codes, and a trap

CDR-H3 varies in length from about 3 to over 30 residues. A fixed numbering scheme has to absorb that
somehow. The answer is **insertion codes**: extra residues get a letter suffix — 111A, 111B, 111C —
so the surrounding positions keep their canonical numbers.

Let's look at a **nanobody** (VHH, a camelid single-domain antibody) with a long CDR3. We will pull it
straight from the PDB.

In [ ]:
from Bio.PDB import PDBParser
from Bio.PDB.Polypeptide import three_to_index, index_to_one


def chain_sequence(pdb_id, chain_id):
    """Download a PDB entry and return the one-letter sequence of one chain."""
    path = f"{pdb_id}.pdb"
    if not os.path.exists(path):
        urllib.request.urlretrieve(f"https://files.rcsb.org/download/{pdb_id}.pdb", path)
    model = PDBParser(QUIET=True).get_structure(pdb_id, path)[0]
    seq = ""
    for r in model[chain_id]:
        if r.id[0] != " ":
            continue
        try:
            seq += index_to_one(three_to_index(r.get_resname()))
        except Exception:
            seq += "X"
    return seq


# 1MEL: cAb-Lys3, a camelid nanobody bound to lysozyme.
VHH = chain_sequence("1MEL", "A")
print(f"Nanobody, {len(VHH)} residues:\n{VHH}\n")

numbering, _ = number(VHH, scheme="imgt")
cdr3_positions = [f"{p}{ic.strip()}" for (p, ic), a in numbering if a != "-" and 105 <= p <= 117]
cdr3_residues  = [a for (p, ic), a in numbering if a != "-" and 105 <= p <= 117]

print(f"IMGT CDR-H3 ({len(cdr3_residues)} residues): {''.join(cdr3_residues)}\n")
print("Position labels, in the order ANARCI returns them:")
print(cdr3_positions)

Nanobody, 132 residues:
VQLQASGGGSVQAGGSLRLSCAASGYTIGPYCMGWFRQAPGKEREGVAAINMGGGITYYADSVKGRFTISQDNAKNTVYLLMNSLEPEDTAIYYCAADSTIYASYYECGHGLSTGGYGYDSWGQGTQVTVSS

IMGT CDR-H3 (26 residues): AADSTIYASYYECGHGLSTGGYGYDS

Position labels, in the order ANARCI returns them:
['105', '106', '107', '108', '109', '110', '111', '111A', '111B', '111C', '111D', '111E', '111F', '112G', '112F', '112E', '112D', '112C', '112B', '112A', '112', '113', '114', '115', '116', '117']


### Look hard at that list of position labels.

```
111, 111A, 111B, 111C, 111D, 111E, 111F, 112G, 112F, 112E, 112D, 112C, 112B, 112A, 112
```

The 111 series counts **up**. The 112 series counts **down**. IMGT inserts residues symmetrically
outward from the middle of the loop, so that the residues nearest each end keep stable numbers
regardless of loop length — which is what makes positions comparable across antibodies of different
H3 length.

**The trap:** if you take these residues and `sort()` them by (number, letter), you will silently
reverse part of your own CDR-H3. Sorting gives `112, 112A, 112B, ... 112G`; the true order is the
reverse. Any code that rebuilds a sequence from sorted numbering is wrong and will not raise an error.

Demonstrate it to yourself.

In [ ]:
true_order = "".join(cdr3_residues)

naive = sorted(
    [((p, ic), a) for (p, ic), a in numbering if a != "-" and 105 <= p <= 117],
    key=lambda x: (x[0][0], x[0][1])        # sort by number, then insertion letter
)
naive_order = "".join(a for _, a in naive)

print("ANARCI order :", true_order)
print("naive sort   :", naive_order)
print("identical?   :", true_order == naive_order)
print("\nRule: trust the order ANARCI gives you. Never re-sort a numbered antibody.")

ANARCI order : AADSTIYASYYECGHGLSTGGYGYDS
naive sort   : AADSTIYASYYECGGTSLGHGYGYDS
identical?   : False

Rule: trust the order ANARCI gives you. Never re-sort a numbered antibody.


## 6. Germlines and somatic hypermutation

ANARCI also reports which **germline genes** a sequence most likely came from, and how close it still
is to them.

Every antibody starts as a recombined V, D and J gene segment from the germline. Then, during affinity
maturation, somatic hypermutation introduces point mutations across the variable domain. **V-gene
identity** — the percentage match to the closest germline V — is therefore a rough readout of how
heavily matured a sequence is.

This matters later: in **week 9** you will humanise a murine antibody, and the whole exercise is about
moving a sequence closer to *human* germline without breaking binding. You need to be able to measure
that distance before you can shrink it.

Let's build a small panel from the PDB and characterise every chain in it.

In [ ]:
PANEL = {
    "trastuzumab_H": ("1N8Z", "B"),   # anti-HER2, humanised
    "trastuzumab_L": ("1N8Z", "A"),
    "pertuzumab_H":  ("1S78", "D"),   # anti-HER2, different epitope (week 1 exercise 3)
    "pertuzumab_L":  ("1S78", "C"),
    "b12_H":         ("1HZH", "H"),   # anti-HIV gp120, broadly neutralising
    "b12_L":         ("1HZH", "L"),
    "CR6261_H":      ("3GBN", "H"),   # anti-influenza haemagglutinin
    "CR6261_L":      ("3GBN", "L"),
    "4G6K_H":        ("4G6K", "H"),   # murine Fab
    "4G6K_L":        ("4G6K", "L"),
    "nanobody":      ("1MEL", "A"),   # camelid VHH
}

sequences = [(name, chain_sequence(pid, cid)) for name, (pid, cid) in PANEL.items()]
numbered, details, _ = anarci(sequences, scheme="imgt", assign_germline=True)

rows = []
for (name, seq), det in zip(sequences, details):
    if not det:
        rows.append({"antibody": name, "chain": "NOT AN ANTIBODY"})
        continue
    d = det[0]
    v_species, v_gene = d["germlines"]["v_gene"][0]
    v_identity = d["germlines"]["v_gene"][1]
    j_species, j_gene = d["germlines"]["j_gene"][0]
    cdr3 = get_cdrs(seq[d["query_start"]:d["query_end"] + 1], "imgt")["CDR3"]
    rows.append({
        "antibody": name, "chain": d["chain_type"], "species": v_species,
        "V_gene": v_gene, "V_identity": round(v_identity, 3),
        "J_gene": j_gene, "CDR3_len": len(cdr3), "CDR3": cdr3,
    })

df = pd.DataFrame(rows)
df

,antibody,chain,species,V_gene,V_identity,J_gene,CDR3_len,CDR3
0,trastuzumab_H,H,human,IGHV3-66*01,0.825,IGHJ4*01,13,SRWGGDGFYAMDY
1,trastuzumab_L,K,human,IGKV1-39*01,0.870,IGKJ1*01,9,QQHYTTPPT
2,pertuzumab_H,H,human,IGHV3-23*04,0.776,IGHJ4*01,12,ARNLGPSFYFDY
3,pertuzumab_L,K,human,IGKV1-33*01,0.859,IGKJ1*01,9,QQYYIYPYT
4,b12_H,H,human,IGHV1-3*01,0.796,IGHJ6*03,20,ARVGPYSWDDSPQDNYYMDV
5,b12_L,K,human,IGKV3-20*01,0.796,IGKJ2*01,9,QVYGASSYT
6,CR6261_H,H,human,IGHV1-69*01,0.847,IGHJ6*03,14,AKHMGYQVRETMDV
7,CR6261_L,L,human,IGLV1-51*01,0.946,IGLJ2*01,12,ATWDRRPTAYVV
8,4G6K_H,H,mouse,IGHV8-4-8*01,0.840,IGHJ3*01,12,ARNRYDPPWFVD
9,4G6K_L,K,mouse,IGKV10-96*01,0.848,IGKJ1*01,9,LQGKMLPWT


### YOUR TURN — Question 2

From the table above:

1. Which chain has the **highest** V-gene identity to germline, and which the **lowest**? What does a
   low value tell you about that antibody's history?
2. `4G6K` is a murine antibody. How did ANARCI know? What would you have to change about it to make it
   a therapeutic candidate for humans?
3. Compare `CDR3_len` across the panel. Which antibody has the longest? Does the ranking match what you
   would guess from the targets these antibodies bind?
4. **One row of this table is wrong.** Not marginally wrong — wrong about the animal the antibody came
   from. Find it. The only clue available to you is in the numbers you already have.

*Double-click and write your answers here.*

1.Highest V-gene identity: CR6261_L (row 7) at 0.946 (94.6%).   Lowest V-gene identity: nanobody (row 10) at 0.714 (71.4%).
2. Fuse the engineered variable domains to human constant regions (e.g., human IgG1 heavy and light chain to prevent immunogenic anti-drug antibody (HAMA) responses.
3.Longest CDR3: nanobody at 26 residues
4.The incorrect row: Row 10 (nanobody).It only has a single H chain (no corresponding light chain) with an exceptionally long CDR3 length (26 residues).

### The wrong row

It is the nanobody. ANARCI reports it as **human `IGHV3-23*01`**, and it is a **camelid** antibody —
it comes from an alpaca or a llama, not a person.

The only clue was the identity score: **0.714**, comfortably the lowest in the panel. Everything else
sits between 0.78 and 0.95. Nothing errored. Nothing warned. The table just quietly contained a false
statement about biology.

Here is why. ANARCI's `allowed_species` parameter defaults to **`['human', 'mouse']`**. It never
considered a camelid germline, because we never asked it to. So it answered the question we actually
asked — *"which human or mouse V gene is closest to this sequence?"* — and answered it correctly. Our
mistake was thinking we had asked a different question.

Run it again with the species set opened up.

**A second trap, worth knowing before you hit it.** The obvious fix is to hand ANARCI a list of species
you care about — `["human", "mouse", "alpaca", "pig", ...]`. Do that and **every light chain in the
panel silently loses its germline assignment**, because some of those species have no light-chain
germline data in ANARCI's database. You would fix one wrong row by blanking ten others. Passing
`allowed_species=None` — "consider everything you know" — is the safe form.

In [ ]:
# allowed_species=None means "consider every species ANARCI knows about".
_, details_all, _ = anarci(sequences, scheme="imgt", assign_germline=True,
                           allowed_species=None)


def v_gene_str(det):
    """Format a V-gene assignment, tolerating a missing one."""
    if not det:
        return "no domain found"
    g = det[0]["germlines"].get("v_gene")
    if not g:
        return "NO ASSIGNMENT"
    (species, gene), identity = g
    return f"{species}/{gene} ({identity:.2f})"


print(f"{'antibody':<16}{'default (human, mouse)':<34}{'all species'}")
print("-" * 78)
for (name, seq), d_def, d_all in zip(sequences, details, details_all):
    a, b = v_gene_str(d_def), v_gene_str(d_all)
    print(f"{name:<16}{a:<34}{b}{'   <-- changed' if a != b else ''}")

antibody        default (human, mouse)            all species
------------------------------------------------------------------------------
trastuzumab_H   human/IGHV3-66*01 (0.82)          human/IGHV3-66*01 (0.82)
trastuzumab_L   human/IGKV1-39*01 (0.87)          human/IGKV1-39*01 (0.87)
pertuzumab_H    human/IGHV3-23*04 (0.78)          human/IGHV3-23*04 (0.78)
pertuzumab_L    human/IGKV1-33*01 (0.86)          human/IGKV1-33*01 (0.86)
b12_H           human/IGHV1-3*01 (0.80)           human/IGHV1-3*01 (0.80)
b12_L           human/IGKV3-20*01 (0.80)          human/IGKV3-20*01 (0.80)
CR6261_H        human/IGHV1-69*01 (0.85)          human/IGHV1-69*01 (0.85)
CR6261_L        human/IGLV1-51*01 (0.95)          human/IGLV1-51*01 (0.95)
4G6K_H          mouse/IGHV8-4-8*01 (0.84)         mouse/IGHV8-4-8*01 (0.84)
4G6K_L          mouse/IGKV10-96*01 (0.85)         mouse/IGKV10-96*01 (0.85)
nanobody        human/IGHV3-23*01 (0.71)          alpaca/IGHV3-3*01 (0.82)   <-- changed


**The nanobody moves to `alpaca/IGHV3-3*01` at 0.82 identity, and nothing else in the panel changes.**

Three things to take from this, and they apply well beyond ANARCI:

1. **A default is an unstated assumption.** Someone decided most users want human or mouse. That is
   probably right, and it was still wrong for us.
2. **The tool did not fail.** There was no error, no warning, no null. A confident, well-formatted,
   wrong answer is the most dangerous output a tool can produce, and you will meet several more this
   term — in week 5 when an epitope predictor is fed a predicted structure, and in week 6 when a
   co-folding model reports high confidence in a wrong complex.
3. **The sanity check was available in the output.** 0.714 was anomalous. You did not need outside
   knowledge to become suspicious, only the habit of looking at the distribution before trusting a row.

One more thing worth noticing: ANARCI calls the nanobody `chain_type = H`. A camelid VHH really is a
heavy-chain variable domain — it binds antigen with **no light chain at all**, using just three CDRs
instead of six. Keep that in mind for week 12, where nanobodies are the *de novo* design target
precisely because a single domain is a far easier thing to design than a paired one.

## 7. The payoff — which scheme should you actually use?

So far this is bookkeeping. Here is the part that decides something.

In week 1 you **measured** trastuzumab's paratope: the 20 residues within 4.5 Å of HER2 — after
throwing out the water molecule that your first attempt mistook for a contact. That was an empirical
fact about the structure, computed with no reference to any numbering scheme.

Now we can ask a real question: **which scheme's CDR definition best captures the residues that are
actually doing the binding?**

Two ways to be wrong:
- a CDR definition that **misses** real contact residues (poor recall)
- a CDR definition so wide that most of what it includes touches nothing (poor precision)

Below are the week 1 paratope residues, as positions in the full chain.

In [ ]:
# Paratope residues from week 1 (the corrected, protein-atoms-only set),
# as 1-based positions in the full PDB chain.
PARATOPE = {"L": [28, 30, 31, 32, 50, 53, 91, 92, 93, 94],
            "H": [33, 50, 52, 57, 58, 59, 99, 103, 104, 105]}
N_PARATOPE = sum(len(v) for v in PARATOPE.values())

FULL_H = chain_sequence("1N8Z", "B")
FULL_L = chain_sequence("1N8Z", "A")


def cdr_positions_in_chain(full_seq, scheme):
    """Which positions of the full chain (1-based) fall inside a CDR under this scheme?"""
    numbering, chain_type = number(full_seq, scheme=scheme)
    ctype = "H" if chain_type == "H" else "L"
    occupied = [((p, ic), a) for (p, ic), a in numbering if a != "-"]
    inside = set()
    for i, ((p, ic), a) in enumerate(occupied, start=1):   # V domain starts at residue 1 here
        for lo, hi in REGIONS[scheme][ctype].values():
            if lo <= p <= hi:
                inside.add(i)
    return inside


results = []
for scheme in ["imgt", "kabat", "chothia"]:
    captured = missed = cdr_total = 0
    missed_list = []
    for ctype, full in [("L", FULL_L), ("H", FULL_H)]:
        cdr_set = cdr_positions_in_chain(full, scheme)
        cdr_total += len(cdr_set)
        for pos in PARATOPE[ctype]:
            if pos in cdr_set:
                captured += 1
            else:
                missed += 1
                missed_list.append(f"{ctype}{pos}")
    results.append({
        "scheme": scheme,
        "paratope_captured": f"{captured}/{N_PARATOPE}",
        "missed": ", ".join(missed_list) or "-",
        "CDR_positions": cdr_total,
        "precision_%": round(100 * captured / cdr_total),
    })

pd.DataFrame(results)

,scheme,paratope_captured,missed,CDR_positions,precision_%
0,imgt,17/20,"L53, H50, H59",47,36
1,kabat,20/20,-,60,33
2,chothia,16/20,"H33, H50, H58, H59",51,31


### What that table says

There is **no winner**, and that is the lesson.

- **Kabat** catches every single contact residue — perfect recall — but it does so by declaring 60
  positions to be CDR. Two thirds of what it calls CDR touches nothing at all.
- **IMGT** is the most precise of the three, but it *misses* three real contact residues. If you used
  IMGT CDRs as your redesign region, you would leave genuine paratope residues untouched.
- **Chothia** has the worst recall here, missing four.

Notice that the two ways of being wrong pull in opposite directions. You cannot fix recall by widening
a definition without paying for it in precision, and the right trade depends entirely on what you are
about to do. Widening is cheap if you are choosing residues to *examine*; it is expensive if you are
choosing residues to *mutate*.

**The practical rule.** Pick the scheme your downstream tool expects — IMGT for most modern tools,
Kabat when working with older literature or patents — and then *do not trust the CDR definition to
tell you what the binding site is.* If you need the binding site, compute it, as you did in week 1.

Beware of concluding too much from one antibody. This is n = 1. Exercise 3 asks you to widen it.

## 8. Deliverable

Run the summary cell, then write the paragraph.


Must contain: the scheme-comparison table, your answers to Questions 1 and 2, the paratope-capture
table, and the paragraph below.

In [ ]:
print("=" * 64)
print("WEEK 2 SUMMARY - trastuzumab numbering")
print("=" * 64)
for scheme in ["imgt", "kabat", "chothia"]:
    h = get_cdrs(VH, scheme)
    print(f"{scheme:<9} CDR-H1 {h['CDR1']:<10} CDR-H2 {h['CDR2']:<18} CDR-H3 {h['CDR3']}")
print()
print(f"H52 is  " + "  ".join(f"{s}={residue_at(VH, s, 52)}" for s in ["imgt", "kabat", "chothia"]))
print(f"Week 1 used: kabat (unstated)")
print("=" * 64)

WEEK 2 SUMMARY - trastuzumab numbering
imgt      CDR-H1 GFNIKDTY   CDR-H2 IYPTNGYT           CDR-H3 SRWGGDGFYAMDY
kabat     CDR-H1 DTYIH      CDR-H2 RIYPTNGYTRYADSVKG  CDR-H3 WGGDGFYAMDY
chothia   CDR-H1 GFNIKDT    CDR-H2 YPTNGY             CDR-H3 WGGDGFYAMDY

H52 is  imgt=W  kabat=Y  chothia=Y
Week 1 used: kabat (unstated)


### Your paragraph

*Double-click and write 150–250 words.*

You are writing to a colleague who has sent you a list of positions to mutate in an antibody, labelled
only `H31, H52, H99`, with no scheme stated. Explain, using numbers you generated today:

- what could go wrong if you guess the scheme
- what specifically differs at those positions between schemes
- what you would ask them for
- and what *you* will do differently when you send positions to someone else

Do not write a general essay about numbering schemes. Write the reply to the colleague.

---

**Your answer:** Hi,

I noticed that the mutation list contains H31, H52, and H99, but the numbering scheme is not specified. This could cause a serious error because the same position number does not necessarily refer to the same residue in different antibody numbering schemes. In the trastuzumab VH sequence we analysed, H31 is D in IMGT, T in Kabat, and W in Chothia. Similarly, H99 is W in IMGT but G in both Kabat and Chothia. Interestingly, H52 is Y in all three schemes, so the ambiguity does not always change the actual amino acid.

If I guessed the scheme, I could therefore mutate the wrong residue, especially at H31 or H99. Before making any changes, could you please tell me which numbering scheme you used for the positions—IMGT, Kabat, or Chothia? If possible, please also provide the antibody sequence or the original numbering output so I can verify the positions.

When I send mutation positions in the future, I will always state the numbering scheme explicitly, for example, “H31 (IMGT)”, rather than simply writing “H31.” This should make the mutation list unambiguous and prevent errors.

## 9. Take-home exercises

Due before week 3.

**Exercise 1 — a conversion table (required).**
Write a function `convert_position(seq, position, from_scheme, to_scheme)` that takes a position label
in one scheme and returns the equivalent label in another, for a given sequence. Test it by converting
Kabat H52 to IMGT and confirming you get back the residue you expect.
*Hint: number the same sequence under both schemes and align by residue index — there is no arithmetic
shortcut, which is the point.*

**Exercise 2 — CDR-H3 length distribution (required).**
Using the panel from section 6, plot CDR-H3 length under IMGT and under Kabat for all heavy chains.
Do the two schemes agree on *which* antibody has the longest H3, even where they disagree on the
absolute length? Comment in one or two sentences on why that matters for a length-based filter.


Both schemes agree completely on the rank order across all antibodies.

    Relative rankings are preserved because IMGT lengths are a constant linear shift from Kabat lengths.

    Hard threshold filters will fail if applied without adjusting for the chosen scheme.

**Exercise 3 — does the week 7 conclusion generalise? (stretch).**
Section 7 compared schemes against one antibody's paratope. Repeat it for **pertuzumab** (`1S78`,
chains C and D, antigen chain A or B — check which). You will need to compute the paratope yourself
using your week 1 code. Does Kabat still have the best recall? Does the same scheme still win?
Report n = 2 honestly: two antibodies is not a benchmark either.

Antibody chains are C (heavy) and D (light).

Antigen partner is HER2 chain A.

Kabat still achieves the highest recall.

Kabat includes wide flanking framework regions, especially in CDR-H2.

Chothia and IMGT offer higher precision by excluding non-contacting loop residues.

Two antibodies are not enough to establish a general rule across all antibody classes.

In [ ]:
# Exercise 1 - scheme conversion. Starter.
#
# The two numberings cover the same residues in the same order, so index i in one
# corresponds to index i in the other. Build both lists, find the index, read across.

def convert_position(seq, position, from_scheme, to_scheme, insertion=" "):
    """Convert a position label between numbering schemes for one sequence."""
    # YOUR CODE HERE
    pass


# print(convert_position(VH, 52, "kabat", "imgt"))

In [ ]:
# Exercise 2 - CDR-H3 length under two schemes.
# You already have `sequences` and `get_cdrs` from section 6.

# YOUR CODE HERE

---

### Where we go next

| Week | Builds on today |
|---|---|
| **3** | You can now name any residue unambiguously. Week 3 uses that to compare paratopes *across* antibodies — impossible without a shared coordinate system. |
| **4** | Structure predictors report per-residue confidence. Confidence at *which* residue? Numbering is how you line predictions up against truth. |
| **9** | Humanisation is measured as movement toward human germline — the V-identity number you computed today. |
| **11** | AntiFold asks you to select an IMGT region to redesign. Today you learned what that selection includes and excludes. |

Before week 3: finish exercises 1 and 2, and commit.

**One habit to start now.** Every time you write down an antibody position, write the scheme next to
it. `H52 (Kabat)`. It prevents a class of error that is very hard to detect
after the fact.